# Testing PlotKit Multi-Provider Support

This notebook demonstrates advanced multi-provider features in PlotKit.

## Features Tested
- Provider-agnostic API usage
- Model registry and metadata
- Provider costs and performance metrics
- Model validation and error handling
- Provider variant system (e.g., groq_default vs groq_openai)

## Setup: Import Libraries

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from plotsense.core.registry_loader import get_registry_loader
from plotsense.explanations.explanations import explainer

load_dotenv()
print("✅ Libraries imported successfully")

## Part 1: Provider Registry System

Explore the dynamic registry that loads provider metadata from GitHub

In [ ]:
# Get the registry loader (singleton)
registry = get_registry_loader()

print("🔍 REGISTRY INFORMATION\n")
print(f"Registry version: {registry.registry.get('version')}")
print(f"Last updated: {registry.registry.get('lastUpdated')}")

# Get all providers
providers_data = registry.registry.get('providers', {})
print(f"\n📦 PROVIDERS ({len(providers_data)} total):")
for provider_name in providers_data.keys():
    variants = providers_data[provider_name].get('variants', {})
    print(f"   - {provider_name}: {list(variants.keys())}")

## Part 2: Available Models by Provider

In [ ]:
# Create a summary of all available models
print("📋 AVAILABLE MODELS BY PROVIDER AND VARIANT\n")

providers = ["openai", "groq", "anthropic", "gemini", "azure", "ollama"]

for provider in providers:
    print(f"\n{'='*50}")
    print(f"{provider.upper()}")
    print(f"{'='*50}")
    
    provider_data = providers_data.get(provider, {})
    variants = provider_data.get('variants', {})
    
    for variant_name, variant_info in variants.items():
        models = variant_info.get('models', [])
        print(f"\n  Variant: {variant_name}")
        for model in models:
            print(f"    - {model}")
        print(f"  Total: {len(models)} models")

## Part 3: Model Costs and Performance

In [ ]:
# Get cost and performance data
print("💰 MODEL COSTS AND PERFORMANCE\n")

# Cost information
costs = registry.get_model_costs()
if costs:
    print("Cost per 1M tokens (input/output):")
    for provider, provider_costs in sorted(costs.items()):
        print(f"\n  {provider}:")
        if isinstance(provider_costs, dict):
            for model, cost_info in sorted(provider_costs.items()):
                if isinstance(cost_info, dict):
                    input_cost = cost_info.get('input', 'N/A')
                    output_cost = cost_info.get('output', 'N/A')
                    print(f"    {model}: ${input_cost} / ${output_cost}")
                else:
                    print(f"    {model}: ${cost_info}")

# Performance data
performance = registry.get_model_performance()
if performance:
    print("\n\nPerformance Scores (0-100):")
    for provider, perf_data in sorted(performance.items()):
        print(f"\n  {provider}:")
        if isinstance(perf_data, dict):
            for model, score in sorted(perf_data.items()):
                print(f"    {model}: {score}")

## Part 4: Provider Variants Explanation

In [ ]:
print("🔄 UNDERSTANDING PROVIDER VARIANTS\n")

print("Some providers have MULTIPLE VARIANTS:")
print()

# Groq has two variants
print("GROQ (2 variants):")
print("  1. groq_default")
print("     - Uses Groq's native API")
print("     - Full access to Groq features")
print()
print("  2. groq_openai")
print("     - OpenAI-compatible interface")
print("     - Compatible with OpenAI clients")
print()

# OpenAI has two variants
print("OPENAI (2 variants):")
print("  1. openai_chat")
print("     - Chat completion models (gpt-4, gpt-3.5-turbo)")
print("     - Optimized for conversation")
print()
print("  2. openai_response")
print("     - Response models (gpt-4o, gpt-4o-mini)")
print("     - Latest vision and multimodal capabilities")
print()

# Others have one variant (default)
print("OTHER PROVIDERS (1 variant each):")
print("  - anthropic_default")
print("  - gemini_default")
print("  - azure_default")
print("  - ollama_default")
print()

print("📌 KEY POINT:")
print("When you specify selected_models=[('openai', 'gpt-4o-mini')],")
print("PlotKit automatically expands it to include BOTH:")
print("  - ('openai_chat', 'gpt-4o-mini')")
print("  - ('openai_response', 'gpt-4o-mini')")

## Part 5: Create Test Data and Plot

In [ ]:
# Create test plot
fig, ax = plt.subplots(figsize=(10, 6))

# Data
x = list(range(2020, 2026))
openai_cost = [0.01, 0.012, 0.008, 0.005, 0.003, 0.002]
groq_cost = [0.0005, 0.0005, 0.0005, 0.0005, 0.0004, 0.0004]
anthropic_cost = [0.008, 0.008, 0.007, 0.006, 0.005, 0.004]

ax.plot(x, openai_cost, marker='o', label='OpenAI', linewidth=2)
ax.plot(x, groq_cost, marker='s', label='Groq', linewidth=2)
ax.plot(x, anthropic_cost, marker='^', label='Anthropic', linewidth=2)

ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Cost per 1M Tokens ($)', fontsize=12)
ax.set_title('LLM Provider Cost Trends (2020-2025)', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("✅ Test plot created")

## Part 6: Test Error Handling - Invalid Model

In [ ]:
# Setup API keys
api_keys = {}
if os.getenv("OPENAI_API_KEY"):
    api_keys["openai"] = os.getenv("OPENAI_API_KEY")

# Test 1: Invalid model name
print("Test 1: Invalid Model Name\n")
print("Attempting: selected_models=[('openai', 'gpt-99-invalid')]")
print()

try:
    result = explainer(
        fig,
        prompt="Test",
        api_keys=api_keys,
        selected_models=[("openai", "gpt-99-invalid")],
        interactive=False
    )
except ValueError as e:
    print("✅ Error caught (as expected):")
    print(f"\n{e}")
    print(f"\n📌 Notice: Error shows the exact model that's wrong")
    print(f"           AND lists all supported models for that provider")

## Part 7: Test Error Handling - Invalid Provider

In [ ]:
# Test 2: Invalid provider name
print("\nTest 2: Invalid Provider Name\n")
print("Attempting: selected_models=[('nonexistent-ai', 'model-xyz')]")
print()

try:
    result = explainer(
        fig,
        prompt="Test",
        api_keys=api_keys,
        selected_models=[("nonexistent-ai", "model-xyz")],
        interactive=False
    )
except ValueError as e:
    print("✅ Error caught (as expected):")
    print(f"\n{e}")
    print(f"\n📌 Notice: Error lists all SUPPORTED providers")

## Part 8: Query Different Providers

In [ ]:
# Test querying different providers
print("\n" + "="*60)
print("TESTING DIFFERENT PROVIDERS")
print("="*60 + "\n")

providers_to_test = [
    ("openai", "gpt-4o-mini", "OpenAI"),
    ("groq", "llama-3.1-8b-instant", "Groq"),
    ("anthropic", "claude-3-haiku-20240307", "Anthropic"),
]

for provider, model, display_name in providers_to_test:
    print(f"\nTesting {display_name}...")
    print(f"  Provider: {provider}")
    print(f"  Model: {model}")
    
    # Check if API key is available
    has_key = provider in api_keys
    print(f"  API Key available: {'✅ Yes' if has_key else '❌ No'}")
    
    if has_key:
        try:
            result = explainer(
                fig,
                prompt="Describe this plot in one sentence.",
                api_keys=api_keys,
                selected_models=[(provider, model)],
                interactive=False,
                timeout=30
            )
            print(f"  Status: ✅ SUCCESS")
            print(f"  Result: {result[:100]}...")
        except Exception as e:
            print(f"  Status: ❌ FAILED")
            print(f"  Error: {str(e)[:80]}...")
    else:
        print(f"  Status: ⏭️  SKIPPED (no API key)")

## Summary

This notebook demonstrated:
- ✅ Registry system with dynamic metadata loading
- ✅ Available models across all providers
- ✅ Cost and performance metrics
- ✅ Provider variant system
- ✅ Comprehensive error handling
- ✅ Cross-provider testing

### Key Takeaways

1. **Registry is Dynamic**: Loaded from GitHub, cached locally, with bundled fallback
2. **Provider Variants**: Some providers have multiple implementations
3. **Smart Expansion**: User specifies `("openai", model)`, system handles all variants
4. **Clear Errors**: Invalid models show supported options
5. **Cost-Aware**: Access cost and performance data for each model
6. **Multi-Provider**: Query any provider with same API

### Next Steps
- Integrate with your data pipeline
- Add your custom visualization logic
- Monitor provider performance and costs
- Use `selected_models` to optimize for cost or speed